In [1]:
# ====================== 04_evaluation.ipynb - 一鍵跑全部 10 個實驗 ======================
# Notebook 專用版本（已修正 __file__ 問題）

import torch
import sys
from pathlib import Path
import os

In [2]:
# ====================== 設定專案根目錄（Notebook 專用） ======================
# 因為你在 /notebooks/ 資料夾，所以 project_root 是上一層
project_root = Path.cwd().parent          # ← 這一行就是關鍵修正！
sys.path.insert(0, str(project_root))

print(f" Project root 已設定為: {project_root}")

from src.config import Config
from src.utils import create_experiment, set_seed
from src.train import train
from src.data import EllipticDataset
from src.split import split_data

 Project root 已設定為: /Users/mankin/Desktop/Anomaly_Detection_on_Graph


/opt/anaconda3/envs/GAD/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# ====================== 統一參數（只需改這裡） ======================
COMMON_PARAMS = {
    "hidden_dim": 32,          # 先用 64 快速測試，正式報告可改 128
    "num_layers": 3,
    "dropout": 0.45,
    "lr": 0.002,
    "epochs": 10,             # 先用 100 測試，正式可改 200
    "patience": 25,          # 早停 patience，正式可改 50

    "use_degree": False,
    "use_pagerank": False,
    "use_clustering": False,
    "use_eigenvector": False,
    "use_betweenness": False,  # 太慢，建議先關閉

    "concat_features": True,
}

In [4]:
# ====================== 所有要跑的 10 個實驗 ======================
experiments = [   # (Model name, Pipeline, Experiment suffix)
    #("GraphSAGE", True,  "sage_pipeline"),
    #("GraphSAGE", False, "sage_e2e"),
    #("FastGCN",   True,  "fastgcn_pipeline"),
    #("FastGCN",   False, "fastgcn_e2e"),
    #("EvolveGCN", True,  "evolvegcn_pipeline"),
    #("EvolveGCN", False, "evolvegcn_e2e"),
    ("DGT",       True,  "dgt_pipeline"),
    #("DGT",       False, "dgt_e2e"),
    #("GAT",       True,  "gat_pipeline"),
    #("GAT",       False, "gat_e2e"),
]

In [5]:
# ====================== 開始執行 ======================
set_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"使用裝置：{device}\n")

# 載入資料（只需執行一次）
print("正在載入 Elliptic 資料集...")
dataset = EllipticDataset(
    root=str(project_root / "data"),
    use_degree=COMMON_PARAMS["use_degree"],
    use_pagerank=COMMON_PARAMS["use_pagerank"],
    use_eigenvector=COMMON_PARAMS["use_eigenvector"],
    use_betweenness=COMMON_PARAMS["use_betweenness"]
)
data = dataset[0]

 Random seed 已固定為 42，所有隨機性已關閉，實驗結果可完全重現。
使用裝置：cpu

正在載入 Elliptic 資料集...


/opt/anaconda3/envs/GAD/lib/python3.11/site-packages/torch_geometric/data/in_memory_dataset.py:131: UserWarning: Weights only load failed. Please file an issue to make `torch.load(weights_only=True)` compatible in your case. Please use `torch.serialization.add_safe_globals([DataEdgeAttr])` to allowlist this global.
  out = fs.torch_load(path)


In [6]:
x = data.x.to(device)
edge_index = data.edge_index.to(device)
y = data.y.to(device)

train_idx, val_idx, test_idx = split_data(data, y, device, temporal_split=True)

print(f" 資料載入完成！開始跑 {len(experiments)} 個實驗...\n")

使用 Temporal Split（推薦）
訓練: 23915 | 驗證: 5979 | 測試: 16670
 資料載入完成！開始跑 1 個實驗...



In [7]:
for model_name, use_pipeline, suffix in experiments:
    print("=" * 90)
    print(f" 正在執行：{model_name} | {'Pipeline' if use_pipeline else 'End-to-End'}")
    print("=" * 90)

    # === 關鍵修正 1：每次 experiment 都重新固定 seed ===
    set_seed(42)                     # ←←← 這一行最重要
    
    # === 關鍵修正 2：清空 GPU cache ===
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()

    # 建立 config
    cfg = Config()
    cfg.exp_name = f"exp_auto_{suffix}"              # prefix      
    cfg.model_name = model_name
    cfg.use_pipeline = use_pipeline
    cfg.hidden_dim = COMMON_PARAMS["hidden_dim"]
    cfg.num_layers = COMMON_PARAMS["num_layers"]
    cfg.dropout = COMMON_PARAMS["dropout"]
    cfg.lr = COMMON_PARAMS["lr"]
    cfg.epochs = COMMON_PARAMS["epochs"]
    cfg.concat_features = COMMON_PARAMS["concat_features"]

    # 建立 experiment 資料夾
    exp_dir = create_experiment(cfg, description=f"Auto-run: {model_name} {'Pipeline' if use_pipeline else 'End-to-End'}")

    # 初始化 model
    model_name_upper = model_name.upper()
    if model_name_upper == "GRAPHSAGE":
        from src.models import ImprovedGraphSAGE
        model = ImprovedGraphSAGE(in_channels=x.shape[1], hidden_dim=cfg.hidden_dim,
                                  num_layers=cfg.num_layers, dropout=cfg.dropout, aggregator="mean").to(device)
    elif model_name_upper == "GAT":
        from src.models import ImprovedGAT
        model = ImprovedGAT(in_channels=x.shape[1], hidden_dim=cfg.hidden_dim,
                            num_layers=cfg.num_layers, heads=8, dropout=cfg.dropout).to(device)
    elif model_name_upper == "FASTGCN":
        from src.models import FastGCN
        model = FastGCN(in_channels=x.shape[1], hidden_dim=cfg.hidden_dim,
                        num_layers=cfg.num_layers, dropout=cfg.dropout).to(device)
    elif model_name_upper == "EVOLVEGCN":
        from src.models import EvolveGCN
        model = EvolveGCN(in_channels=x.shape[1], hidden_dim=cfg.hidden_dim,
                          num_layers=cfg.num_layers, dropout=cfg.dropout).to(device)
    elif model_name_upper == "DGT":
        from src.models import DGT
        model = DGT(in_channels=x.shape[1], hidden_dim=cfg.hidden_dim,
                    num_layers=cfg.num_layers, heads=4, dropout=cfg.dropout).to(device)
    else:
        raise ValueError(f"未知 model: {model_name}")

    # 執行訓練
    train(model, x, edge_index, y, train_idx, val_idx, test_idx, cfg, device, exp_dir)

    print(f" {model_name} {'Pipeline' if use_pipeline else 'End-to-End'} 完成！\n")

print(" 全部 10 個實驗已自動完成！")
print(f"請到以下資料夾查看所有 results.json：")
print(f"   {project_root / 'experiments'}")

 正在執行：DGT | Pipeline
 Random seed 已固定為 42，所有隨機性已關閉，實驗結果可完全重現。
本次實驗名稱：exp_auto_dgt_pipeline
實驗資料夾：../experiments/exp_auto_dgt_pipeline
config.yaml 已自動建立
實驗設定已儲存至: ../experiments/exp_auto_dgt_pipeline/config.yaml
 DGT Kaiming initialization completed (heads=4, hidden=32)
 切換至 Pipeline 模式 (GNN → XGBoost)
 Using Pipeline for DGT → XGBoost
 Training model: DGT | hidden_dim=32, layers=3, dropout=0.45
 Training started at 2026-03-31 19:40:13

 已 concat 原始 features + GNN embeddings → 新 dimension = 197

 Training finished in 3.3 seconds (0.06 minutes)
results.json 已儲存至: ../experiments/exp_auto_dgt_pipeline/results.json (包含訓練時間紀錄)
已複製最新模型 graphsage_best_20260331_181103.pt → ../experiments/exp_auto_dgt_pipeline/model_best.pt

實驗 exp_auto_dgt_pipeline 所有結果已儲存至：
   ../experiments/exp_auto_dgt_pipeline/
   ├── config.yaml
   ├── results.json
   └── model_best.pt

 實驗完成！
   名稱：exp_auto_dgt_pipeline
   路徑：../experiments/exp_auto_dgt_pipeline
   檔案：
     • config.yaml
     • results.json
     • model_b